# Event Summary EDA: Does NYT cover the GDELT events we pull?

We embed NYT Stage-2 `summary` text and GDELT `event_summary` text with **BGE-large-en-v1.5**, then use cosine similarity to:
1. Inspect the top-10 NYT matches for a chosen GDELT event (qualitative check).
2. For each 2015 GDELT event, count NYT headlines whose cosine similarity exceeds a tunable threshold (coverage check).

Embeddings are cached to `data/embeddings/` so reruns are cheap.

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path(r"C:\Users\gongz\finance_world_model")
NYT_PATH    = PROJECT_ROOT / "data" / "nyt_stage2" / "nyt_stage2_2015.parquet"
GDELT_PATH  = PROJECT_ROOT / "data" / "nyt_stage2" / "gdelt_events_with_market_impact.csv"
EMB_DIR     = PROJECT_ROOT / "data" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID    = "BAAI/bge-large-en-v1.5"
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 32

print(f"Device: {DEVICE}")

Device: cuda


## Load data

In [15]:
nyt_df = pd.read_parquet(NYT_PATH)
gdelt_df_raw = pd.read_csv(GDELT_PATH)

gdelt_df = gdelt_df_raw[gdelt_df_raw['event_year'] == 2015].copy()

print(f"NYT rows:   {len(nyt_df):,}")
print(f"GDELT rows: {len(gdelt_df):,}")
print("\n--- NYT summary sample ---")
print(nyt_df[["date", "headline", "summary"]].head(2).to_string(index=False))
print("\n--- GDELT event_summary sample ---")
print(gdelt_df[["SQLDATE", "Actor1Name", "event_summary"]].head(2).to_string(index=False))

NYT rows:   5,242
GDELT rows: 200

--- NYT summary sample ---
      date                                                headline                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

## Embed with BGE-large-en-v1.5

BGE-large-en-v1.5 is CLS-pooled and we ask `sentence-transformers` to L2-normalize so a plain dot product equals cosine similarity. Cached embeddings are stored under `data/embeddings/` keyed by source file — first run computes, later runs just load.

In [16]:
def embed_or_load(texts, cache_path, model, batch_size=BATCH_SIZE):
    cache_path = Path(cache_path)
    if cache_path.exists():
        print(f"Loading cached embeddings: {cache_path.name}")
        return np.load(cache_path)
    print(f"Computing {len(texts)} embeddings -> {cache_path.name}")
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    np.save(cache_path, emb)
    return emb

In [17]:
nyt_cache   = EMB_DIR / "nyt_stage2_2015_bge.npy"
gdelt_cache = EMB_DIR / "gdelt_2015_bge.npy"

# Only load the (slow) BGE model if at least one cache is missing.
if nyt_cache.exists() and gdelt_cache.exists():
    print("Both embedding caches found — skipping model load.")
    print(f"  {nyt_cache}")
    print(f"  {gdelt_cache}")
    model = None
else:
    missing = [str(p) for p in (nyt_cache, gdelt_cache) if not p.exists()]
    print("Missing cache(s):")
    for m in missing:
        print(f"  {m}")
    print(f"Loading {MODEL_ID} on {DEVICE} ...")
    model = SentenceTransformer(MODEL_ID, device=DEVICE)

nyt_texts   = nyt_df["summary"].fillna("").astype(str).tolist()
gdelt_texts = gdelt_df["event_summary"].fillna("").astype(str).tolist()

nyt_emb   = embed_or_load(nyt_texts,   nyt_cache,   model)
gdelt_emb = embed_or_load(gdelt_texts, gdelt_cache, model)

print("NYT emb:  ", nyt_emb.shape)
print("GDELT emb:", gdelt_emb.shape)

Both embedding caches found — skipping model load.
  C:\Users\gongz\finance_world_model\data\embeddings\nyt_stage2_2015_bge.npy
  C:\Users\gongz\finance_world_model\data\embeddings\gdelt_2015_bge.npy
Loading cached embeddings: nyt_stage2_2015_bge.npy
Loading cached embeddings: gdelt_2015_bge.npy
NYT emb:   (5242, 1024)
GDELT emb: (200, 1024)


## Top-10 NYT matches for a single GDELT event

Pick a GDELT event by row index. We score every NYT summary by cosine similarity to its embedding and return the top-K. Eyeball the headlines/summaries to judge whether the event is plausibly covered.

In [18]:
import textwrap

WRAP_WIDTH = 100

def _wrap(text: str, width: int = WRAP_WIDTH, indent: str = "") -> str:
    return textwrap.fill(
        str(text),
        width=width,
        initial_indent=indent,
        subsequent_indent=indent,
        break_long_words=False,
        replace_whitespace=False,
    )

def top_k_nyt_for_gdelt(gdelt_idx: int, k: int = 10, summary_chars: int = 400):
    query = gdelt_emb[gdelt_idx:gdelt_idx + 1]            # (1, 1024)
    sims  = cosine_similarity(query, nyt_emb).ravel()      # (N_nyt,)
    top   = np.argsort(-sims)[:k]

    g = gdelt_df.iloc[gdelt_idx]
    print("=" * WRAP_WIDTH)
    print(f"GDELT [{gdelt_idx}]  date={g.get('SQLDATE', '')}  actor={g.get('Actor1Name', '')}")
    print("event_summary:")
    print(_wrap(g['event_summary'], indent="  "))
    print("=" * WRAP_WIDTH)

    rows = []
    for rank, i in enumerate(top, 1):
        nyt_row  = nyt_df.iloc[i]
        sim      = float(sims[i])
        date     = nyt_row.get("date", "")
        headline = nyt_row.get("headline", "")
        summary  = str(nyt_row["summary"])[:summary_chars]

        print(f"\n[#{rank}] cos_sim={sim:.4f}  date={date}")
        print(_wrap(f"headline: {headline}", indent="  "))
        print(_wrap(f"summary:  {summary}", indent="  "))

        rows.append({
            "rank": rank,
            "cos_sim": round(sim, 4),
            "date": date,
            "headline": headline,
            "summary_preview": summary[:200],
        })
    return pd.DataFrame(rows)


In [25]:
top_k_nyt_for_gdelt(5, k=10)

GDELT [5]  date=20151007  actor=NIGERIA
event_summary:
  The Nigerian military killed 50 Boko Haram militants and lost nine soldiers during a fierce battle
  in Goniri village, Yobe State, on October 7, 2015. The clash broke out after approximately 200
  insurgents attacked the village, prompting a counter-operation that resulted in the seizure of 16
  vehicles and a significant cache of weaponry.

[#1] cos_sim=0.7554  date=2015-07-17
  headline: Witnesses Detail Nigerian Market Bombing
  summary:  On July 17, 2015, a bombing by suspected Boko Haram militants occurred at a market in
  Gombe, Nigeria, with two witnesses describing the attack. The incident was part of an ongoing
  upsurge in violence by the Islamist group, which has targeted both Muslims and Christians, often
  focusing on individuals perceived as supporting the Nigerian government. This attack follows a
  pattern of Boko Haram's insu

[#2] cos_sim=0.7550  date=2015-02-18
  headline: In Nigeria, Boko Haram Loses Ground t

,rank,cos_sim,date,headline,summary_preview
0,1,0.7554,2015-07-17,Witnesses Detail Nigerian Market Bombing,"On July 17, 2015, a bombing by suspected Boko ..."
1,2,0.7550,2015-02-18,"In Nigeria, Boko Haram Loses Ground to Chadians","Chadian troops entered Nigeria on February 22,..."
2,3,0.7517,2015-12-29,Suspected Boko Haram Attacks Kill Scores in Ni...,At least 52 people were killed and 124 injured...
3,4,0.7513,2015-02-05,Chad Retakes Nigerian Town From Militant Group...,Chad's military recaptured a border town in Ni...
4,5,0.7408,2015-10-28,Nigerian Military Says It Has Rescued Over 300...,The Nigerian military claimed it rescued over ...
5,6,0.7365,2015-03-20,Nigerian Army Noticeably Absent in Town Taken ...,"Boko Haram was driven out of Damasak, Nigeria,..."
6,7,0.7321,2015-02-06,"Boko Haram Widens Fight, Striking Niger","Boko Haram launched an attack on Bosso, a loca..."
7,8,0.7311,2015-03-12,Mercenaries Join Nigeria’s Military Campaign A...,Hundreds of mercenaries from South Africa and ...
8,9,0.7301,2015-12-15,Shiite Muslim Sect Alleges Massacre by Nigeria...,A Shiite Muslim sect in northwestern Nigeria a...
9,10,0.7300,2015-06-03,Abuses by Nigeria’s Military Found to Be Rampa...,Amnesty International reported that Nigerian s...


## Coverage by threshold

For each GDELT event, count the number of NYT headlines whose cosine similarity exceeds a chosen threshold. Aggregate stats give us a sense of how well NYT covers the GDELT set.

In [20]:
gdelt_df.columns

Index(['event_year', 'rank_in_year', 'GLOBALEVENTID', 'SQLDATE', 'Actor1Name',
       'Actor1Type1Code', 'Actor1CountryCode', 'Actor2Name', 'Actor2Type1Code',
       'Actor2CountryCode', 'EventCode', 'EventRootCode', 'QuadClass',
       'ActionGeo_FullName', 'ActionGeo_CountryCode', 'GoldsteinScale',
       'NumArticles', 'AvgTone', 'impact_score', 'SOURCEURL', 'event_summary',
       'key_actors', 'event_time', 'fetch_success', 'market_impact'],
      dtype='str')

In [21]:
len(gdelt_df)

200

In [22]:
THRESHOLD = 0.75  # tweak

# `gdelt_emb` was computed from the full event_year==2015 frame (200 rows).
# Re-derive that frame from gdelt_df_raw so this cell is idempotent regardless
# of how many times it gets re-run, then build a boolean mask aligned to it.
gdelt_df_2015 = gdelt_df_raw[gdelt_df_raw["event_year"] == 2015].reset_index(drop=True)
assert len(gdelt_df_2015) == gdelt_emb.shape[0], (
    f"row mismatch: gdelt_df_2015={len(gdelt_df_2015)}, gdelt_emb={gdelt_emb.shape[0]} "
    "— re-run cells 3 and 6 to regenerate embeddings."
)

# Keep only events flagged as market-impacting; subset df + embeddings together.
mask        = (gdelt_df_2015["market_impact"] == 1).values
gdelt_df_f  = gdelt_df_2015.loc[mask].reset_index(drop=True)
gdelt_emb_f = gdelt_emb[mask]

sim_matrix = cosine_similarity(gdelt_emb_f, nyt_emb)
counts     = (sim_matrix >= THRESHOLD).sum(axis=1)

coverage = pd.DataFrame({
    "gdelt_idx":      np.arange(len(gdelt_df_f)),
    "date":           gdelt_df_f["SQLDATE"].values,
    "actor":          gdelt_df_f["Actor1Name"].values,
    "event_summary":  gdelt_df_f["event_summary"].astype(str).str[:120].values,
    "max_sim":        sim_matrix.max(axis=1).round(4),
    "n_above_thresh": counts,
})

print(f"Threshold = {THRESHOLD}")
print(f"Filtered GDELT events: {len(gdelt_df_f)} (from {len(gdelt_df_2015)})")
print(f"Events with >=1 NYT match: {(counts >= 1).sum()} / {len(counts)}")
print(f"Mean matches per event: {counts.mean():.2f}")
print(f"Median:                  {np.median(counts):.0f}")
print(f"Max:                     {counts.max()}")

coverage.sort_values("n_above_thresh", ascending=False).head(20)

Threshold = 0.75
Filtered GDELT events: 51 (from 200)
Events with >=1 NYT match: 35 / 51
Mean matches per event: 6.53
Median:                  1
Max:                     88


,gdelt_idx,date,actor,event_summary,max_sim,n_above_thresh
6,6,20150602,GREEK,The Greek government submitted a comprehensive...,0.8256,88
47,47,20151124,TURKEY,Turkey shot down a Russian Su-24 fighter jet n...,0.8942,34
29,29,20151115,EUROPE,"On November 15, 2015, France launched massive ...",0.8510,23
7,7,20151215,RUSSIAN,Suspected Russian airstrikes targeted two mark...,0.8675,18
26,26,20151125,TURKEY,"Following the downing of a Russian warplane, T...",0.8587,17
1,1,20150527,SINAI,An Egyptian policeman was killed and eight oth...,0.8494,17
0,0,20150522,IRAQI,Thousands of Shia militia fighters arrived at ...,0.8085,16
43,43,20151114,FRENCH,"On November 14, 2015, French President Françoi...",0.8372,16
17,17,20150313,CHAD,Chadian and Nigerien military forces killed at...,0.8224,11
31,31,20151111,RUSSIAN,Ukrainian military forces repelled two separat...,0.7998,10


In [23]:
# Threshold sensitivity: how does coverage change as we sweep threshold?
for t in [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    n = (sim_matrix >= t).sum(axis=1)
    covered = (n >= 1).sum()
    print(f"thresh={t:.2f}  covered={covered:>4d}/{len(n)}  mean_matches={n.mean():6.2f}  median={int(np.median(n))}")

thresh=0.60  covered=  51/51  mean_matches=521.39  median=432
thresh=0.65  covered=  51/51  mean_matches=142.08  median=102
thresh=0.70  covered=  48/51  mean_matches= 30.10  median=14
thresh=0.75  covered=  35/51  mean_matches=  6.53  median=1
thresh=0.80  covered=  19/51  mean_matches=  1.14  median=0
thresh=0.85  covered=   6/51  mean_matches=  0.22  median=0
thresh=0.90  covered=   0/51  mean_matches=  0.00  median=0
